# 01 — Acquisition des données (GSE5281)

**Objectif de ce notebook :** télécharger le dataset GEO **GSE5281** via `GEOparse`, le mettre en cache localement, et vérifier sa structure de base avant l'exploration approfondie (notebook 02).

## Le dataset

[GSE5281](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5281) (Liang *et al.*, 2007) est une étude d'expression génique par microarray comparant le cerveau de patients atteints de la **maladie d'Alzheimer** à celui de **témoins sains âgés**.

- **Plateforme :** Affymetrix Human Genome U133 Plus 2.0 (GPL570) — ~54 000 sondes.
- **Tissu :** plusieurs régions cérébrales prélevées par micro-dissection laser.
- **Notre périmètre :** nous restreindrons l'analyse au **cortex entorhinal**, la première structure cérébrale atteinte dans la maladie d'Alzheimer. Ce filtrage par région se fera au notebook 02 ; ici on télécharge l'ensemble.

## Note sur la nature des données

GEO distribue les données sous forme de fichier **SOFT** (`.soft.gz`) : un format texte qui regroupe les métadonnées de l'étude, la description de la plateforme (table de correspondance sonde → gène), et les valeurs d'expression de chaque échantillon (GSM). `GEOparse` parse ce fichier en objets Python manipulables.

In [1]:
import sys
from pathlib import Path

# Rendre le package src/ importable depuis le dossier notebooks/.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_loader

print("Racine projet :", PROJECT_ROOT)
print("Cache données :", data_loader.RAW_DIR)

Racine projet : C:\Users\Maxime\Desktop\Projet data science\alzheimer-gene-expression
Cache données : C:\Users\Maxime\Desktop\Projet data science\alzheimer-gene-expression\data\raw


## Téléchargement (avec cache)

`data_loader.load_gse()` télécharge le fichier SOFT de GSE5281 dans `data/raw/` au premier appel, puis réutilise la copie locale ensuite. Le téléchargement initial peut prendre une à quelques minutes selon la connexion (fichier de plusieurs dizaines de Mo).

In [2]:
gse = data_loader.load_gse()
gse

C:\Users\Maxime\venv\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


<SERIES: GSE5281 - 161 SAMPLES, 1 d(s)>

## Vérification de la structure

On inspecte les briques de base de l'objet GSE : métadonnées de l'étude, plateforme(s) utilisée(s), et nombre d'échantillons (GSM). C'est un contrôle de bon téléchargement — l'exploration détaillée (groupes AD/contrôle, régions, QC) se fera au notebook 02.

In [3]:
# Métadonnées de l'étude
print("Titre :", gse.metadata.get("title", ["?"])[0])
print("Résumé :", gse.metadata.get("summary", ["?"])[0][:300], "...\n")

# Plateforme(s)
for gpl_name, gpl in gse.gpls.items():
    print(f"Plateforme {gpl_name} : {gpl.metadata.get('title', ['?'])[0]}")
    print(f"  Table d'annotation : {gpl.table.shape[0]} sondes × {gpl.table.shape[1]} colonnes")

# Échantillons
print(f"\nNombre d'échantillons (GSM) : {len(gse.gsms)}")

Titre : Alzheimer's disease and the normal aged brain (steph-affy-human-433773)
Résumé : Information about the genes that are preferentially expressed during the course of Alzheimerâ€™s disease (AD) could improve our understanding of the molecular mechanisms involved in the pathogenesis of this common cause of cognitive impairment in older persons, provide new opportunities in the diagn ...

Plateforme GPL570 : [HG-U133_Plus_2] Affymetrix Human Genome U133 Plus 2.0 Array
  Table d'annotation : 54675 sondes × 16 colonnes

Nombre d'échantillons (GSM) : 161


In [4]:
# Aperçu d'un échantillon : ses caractéristiques annotées (région, diagnostic...)
# Ces champs serviront au notebook 02 pour séparer AD/contrôle et filtrer le cortex entorhinal.
first_gsm_name = list(gse.gsms.keys())[0]
first_gsm = gse.gsms[first_gsm_name]

print(f"Exemple — {first_gsm_name} :")
print("  Titre :", first_gsm.metadata.get("title", ["?"])[0])
for ch in first_gsm.metadata.get("characteristics_ch1", []):
    print("   -", ch)

# Aperçu de la matrice d'expression de cet échantillon
print("\nTable d'expression (sonde -> valeur) :")
first_gsm.table.head()

Exemple — GSM119615 :
  Titre : EC control 1
   - Sample Amount: 10 ug
   - Bio-Source Name: EC control 1
   - Organism: HumanÂ
   - Organ/Tissue Type: brainÂ
   - Organ Region: Entorhinal CortexÂ
   - Cell Type: layer III neuronsÂ
   - Ethnicity: CaucasianÂ
   - Developmental Stage: AdultÂ
   - Disease State: normalÂ
   - Sex: maleÂ
   - Genetic Variation: NoneÂ
   - Age: 63 years

Table d'expression (sonde -> valeur) :


,ID_REF,VALUE,ABS_CALL,DETECTION P-VALUE
0,AFFX-BioB-5_at,276.48065,P,0.000754
1,AFFX-BioB-M_at,391.85696,P,0.000169
2,AFFX-BioB-3_at,269.12082,P,0.000095
3,AFFX-BioC-5_at,852.27240,P,0.000095
4,AFFX-BioC-3_at,821.93243,P,0.000044
